# CLIFFGUARD — round 5: does any of it hold outside round-to-nearest?

Every measurement in this project so far uses one quantizer family. RTN was
chosen for causal cleanliness — bit-width is the only thing that varies, so a
ladder is an ordinal axis rather than a list of unrelated methods — and that
choice is exactly what makes the results hard to generalise. Nobody deploys RTN.
The reports this paper is arguing with use AWQ, GPTQ and GGUF k-quants.

This round runs the protocol against checkpoints quantized by methods people
actually ship, on the same prompts, against the same full-precision baselines,
with the same gate, the same judge and the same corrected label scorer.

**The protocol was frozen first.** `docs/preregistration_round5.md` states two
confirmatory hypotheses with thresholds a null result can fail, and two
exploratory ones that are labelled as such. The cell below prints its SHA-256
and stores it in the archive, so which version was in force is checkable rather
than asserted. Nothing in it may be edited after this notebook runs.

## The five steps, in priority order

| step | what it answers | time |
|---|---|---|
| 1 | Deployed quantizers: AWQ and GPTQ-Int4 on Qwen2.5-3B | ~30 min |
| 2 | The 48-token window, at the quantized rung rather than only at FP16 | ~55 min |
| 3 | Is greedy nondeterminism a batch-size effect? | ~12 min |
| 4 | Sampled decoding across seeds | ~55 min |
| 5 | Scale: the same three schemes at 7B, if the VRAM is there | ~60 min |

Run all. Each step checkpoints to Drive as it finishes and skips work already
done, so a disconnect costs one step and not the session. Steps 4 and 5 are the
ones to drop if time runs short; step 5 skips itself on a T4.

## Environment

In [ ]:
# One flag for "is this session worth spending GPU time on", set here and
# updated by the preflight below. Whether `raise SystemExit` ends a Colab
# "Run all" is a property of the runtime, and a notebook that runs unattended
# for hours should not depend on the answer -- so every cell that spends GPU
# time opens by checking this, and the raises below are belt to its braces.
SESSION_OK = True

import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/parnish007/CLIFFGUARD.git'
# Set to the commit this notebook was validated against; '' follows main.
REPO_COMMIT = '59dfdaa19454e46bf50e74dea70ad930be4b7fd0'  # set after round 5 is validated; '' follows main
REPO_DIR = pathlib.Path('/content/CLIFFGUARD') if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/cliffguard')

if IN_COLAB:
    # Fatal, not a warning. Without Drive every cache and run directory lives
    # on disk Colab wipes at disconnect, so an unattended session that loses
    # its connection at hour two loses the whole session. Continuing without it
    # is not a degraded run, it is a run that cannot survive the thing most
    # likely to happen to it.
    from google.colab import drive as _drive
    _drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    # Pinned to the commit this notebook was written against. A clone of
    # whatever main happens to be would silently run different analysis code,
    # and --depth 1 cannot reach a specific commit, hence the full clone above.
    if REPO_COMMIT:
        subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], check=True)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                          text=True, check=True).stdout.strip()
    print(f'repo commit  : {head}')
    if REPO_COMMIT and not head.startswith(REPO_COMMIT):
        SESSION_OK = False
        raise SystemExit(f'checked out {head}, expected {REPO_COMMIT}')

    # check=True: a missing bitsandbytes surfaces as a CUDA error inside the
    # judge two hours from now rather than here.
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'bitsandbytes', 'datasets', 'accelerate'], check=True)

    # AWQ and GPTQ checkpoints carry their quantization in the weights, and
    # transformers dispatches to a backend that has to be installed: autoawq for
    # one, optimum + gptqmodel for the other. Neither is in the cell above,
    # which came from round 4 -- a notebook that only grades stored text and
    # needs no quantization backend at all.
    #
    # NOT check=True. These wheels are large, occasionally unavailable for the
    # runtime's torch, and a failure here should cost step 1 rather than the
    # session: steps 2, 3 and 4 answer questions of their own and need none of
    # this. The result is recorded and step 1 reads it.
    DEPLOYED_BACKENDS_OK = subprocess.run(
        [sys.executable, '-m', 'pip', '-q', 'install',
         'autoawq', 'optimum', 'gptqmodel'],
        capture_output=True).returncode == 0
    print(f'AWQ/GPTQ backends: '
          f'{"installed" if DEPLOYED_BACKENDS_OK else "FAILED"}')
else:
    DEPLOYED_BACKENDS_OK = True

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch, numpy as np, transformers
HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else 'NONE'
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0
print(f'repo         : {pathlib.Path.cwd()}')
print(f'python       : {platform.python_version()}')
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {np.__version__}')
print(f'GPU          : {GPU_NAME}  ({VRAM_GB} GB)')
if hasattr(os, 'statvfs'):
    st = os.statvfs('.')
    print(f'free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB')
if not HAS_GPU:
    SESSION_OK = False
    raise SystemExit('No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.')

# Import them, rather than trusting that pip said it was happy. A wheel that
# installs against the wrong torch imports with an OSError, and finding that out
# here costs a second; finding it out inside step 1 costs the fp16 baseline that
# has already been generated by then.
if DEPLOYED_BACKENDS_OK:
    try:
        import awq            # noqa: F401
        import gptqmodel      # noqa: F401
        import optimum        # noqa: F401
    except Exception as _error:                       # noqa: BLE001
        DEPLOYED_BACKENDS_OK = False
        print(f'AWQ/GPTQ backends installed but do not import ({_error}); '
              'steps 1 and 5 will be skipped')
if tuple(int(p) for p in transformers.__version__.split('.')[:2]) < (4, 45):
    SESSION_OK = False
    raise SystemExit(f'transformers {transformers.__version__} too old (need >= 4.45).\nRun: !pip -q install -U transformers, then restart the runtime.')


## Restore, and the frozen protocol

In [ ]:
# Restore Drive state BEFORE validating it. On a fresh Colab clone `data/` does
# not exist -- it is gitignored -- so the corpora can only come from Drive or a
# rebuild, and the run directories from a previous session likewise. Validating
# first would fail on files that were about to arrive, or pass a check on files
# that were never going to.
import shutil

def _restore(src: pathlib.Path, dst: pathlib.Path, what: str) -> bool:
    if not src.exists():
        print(f'[drive] no {what} to restore')
        return False
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[drive] restored {what} -> {dst}')
    return True

_restore(DRIVE_ROOT / 'fold_a', pathlib.Path('data/folds/fold_a'), 'Fold A corpus')
_restore(DRIVE_ROOT / 'eval_suites', pathlib.Path('data/eval_suites'), 'eval suites')
_restore(DRIVE_ROOT / 'artifacts' / 'runs', pathlib.Path('artifacts/runs'),
         'previous run directories')

# The five published runs now ship WITH the repository, so the clone already
# has them and step 3 needs no upload. Verified rather than assumed: their
# absence would make the re-grade skip silently, and that is the step whose
# whole purpose is to check the published numbers.
_published = ['colab-behavioural-qwen3b', 'colab-behavioural-phi35',
              'lab-qwen3b-xstest', 'lab-phi35-xstest', 'lab-smol17-xstest']
_found = [p for p in _published
          if list(pathlib.Path('artifacts/runs').glob(f'*{p}'))]
print(f'[repo] published runs available: {len(_found)}/{len(_published)}')
if len(_found) < len(_published):
    print(f'       MISSING {sorted(set(_published) - set(_found))} -- the '
          'step-3 re-grade will skip those and report them as not run')

# A Drive zip still overrides, for anyone re-running with different data --
# but only under artifacts/runs/, which is the only thing a prior-runs archive
# is for.
#
# extractall('.') was unrestricted. CPython does strip '..' and leading
# separators, so nothing escapes the working directory; what it does not stop is
# a member named scripts/classify_completions_judge.py silently replacing the
# grader, or .git/hooks/post-checkout running on the next git command. Both
# were reproduced in a test. The archive comes from the user's own Drive, so
# this is not an attack anyone is mounting -- it is an integrity hole in a
# project whose whole argument is that the instruments were not edited between
# being written and being run, and it costs four lines to close.
_prior = DRIVE_ROOT / 'prior_runs.zip'
if _prior.exists():
    import zipfile
    _allowed, _refused = [], []
    with zipfile.ZipFile(_prior) as zf:
        for _member in zf.namelist():
            _parts = [p for p in _member.replace(chr(92), '/').split('/')
                      if p not in ('', '.', '..')]
            (_allowed if _parts[:2] == ['artifacts', 'runs']
             else _refused).append(_member)
        zf.extractall('.', members=_allowed)
    print(f'[drive] {_prior.name}: unpacked {len(_allowed)} file(s) under '
          f'artifacts/runs/')
    if _refused:
        print(f'        REFUSED {len(_refused)} member(s) outside it, which a '
              'prior-runs archive has no business carrying:')
        for _member in _refused[:10]:
            print(f'          {_member}')
        if len(_refused) > 10:
            print(f'          ... and {len(_refused) - 10} more')

# Rebuild only what is still missing. Fold A comes from an UNPINNED HuggingFace
# revision, so a rebuild can silently produce a different corpus and break the
# pairing with the 48-token runs; preflight hashes it immediately afterwards,
# which is what turns that risk into a caught error rather than a wasted
# session. XSTest is a single file at a stable URL and is safer to fetch.
if not pathlib.Path('data/folds/fold_a/anthropic_hh_refused.jsonl').exists():
    print('[corpus] Fold A missing; rebuilding (preflight will verify the hash)')
    subprocess.run([sys.executable, 'scripts/download_fold_a.py', '--download'],
                   check=True)
if not pathlib.Path('data/eval_suites/xstest.jsonl').exists():
    print('[corpus] XSTest missing; fetching')
    subprocess.run([sys.executable, 'scripts/download_eval_suites.py',
                    '--download', '--suites', 'xstest'], check=False)

# Cache the corpora back, so the next session restores instead of re-fetching.
if DRIVE_ROOT.exists():
    for src, name in ((pathlib.Path('data/folds/fold_a'), 'fold_a'),
                      (pathlib.Path('data/eval_suites'), 'eval_suites')):
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / name, dirs_exist_ok=True)
    print('[drive] corpora cached')

# The preregistration's hash, printed and carried into the archive. A protocol
# that can be edited between the writing and the running is not frozen, and a
# claim that it was not edited is worth exactly as much as the claim that the
# results were not tuned. This is the cheap way to make both checkable.
import hashlib
PREREG_OK = True
PREREG = pathlib.Path('docs/preregistration_round5.md')
if PREREG.exists():
    PREREG_SHA = hashlib.sha256(PREREG.read_bytes()).hexdigest()
    print(f'\n[prereg] {PREREG} sha256={PREREG_SHA}')
    # Against git, not just against the working tree. This cell runs AFTER the
    # restore above, and a hash of a file something else has just written is a
    # hash of that file -- true, and useless as evidence that the protocol is
    # the one that was committed. The pin makes the checkout reproducible; this
    # makes it checkable that nothing since has edited the one document the
    # round is supposed to be held to.
    _dirty = subprocess.run(['git', 'diff', '--quiet', 'HEAD', '--', str(PREREG)])
    PREREG_OK = _dirty.returncode == 0
    if not PREREG_OK:
        print(f'{PREREG} DIFFERS from the committed version. A preregistration '
              'that can be edited between the writing and the running is not '
              'frozen. Restore it with: git checkout -- ' + str(PREREG))
    else:
        print('[prereg] matches the committed version; git log:')
        subprocess.run(['git', 'log', '--oneline', '--', str(PREREG)])
else:
    PREREG_SHA = None
    PREREG_OK = False
    print('docs/preregistration_round5.md is MISSING. This round is only worth '
          'running against a protocol fixed before it; without the file there '
          'is nothing to be held to.')

## Preflight

In [ ]:
# --require-xstest because this notebook's second half needs the labelled
# corpus. Without the flag a missing file is only a `skip`, preflight passes,
# and the failure surfaces after the two expensive HH-RLHF steps have already
# spent their GPU time.
preflight = subprocess.run([sys.executable, 'scripts/preflight_round2.py',
                            '--require-xstest'])
if preflight.returncode != 0:
    SESSION_OK = False
    print('PREFLIGHT FAILED. Repair the reported gate failure before running '
          'anything below. Every check above runs on CPU in seconds, so fixing '
          'it costs nothing compared with discovering it at hour three.')

## Configuration

In [ ]:
MODEL_3B = 'Qwen/Qwen2.5-3B-Instruct'
MODEL_7B = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_BATCH = 4          # round 3 and round 4 both; see round 4 for why
JUDGE_COMPLETION_CHARS, TAXONOMY_MAX_LENGTH = 2000, 2560
N_LONG, N_XSTEST, SEED = 250, 150, 0
SHORT_TOKENS, LONG_TOKENS = 48, 256

# Deployed quantizers, as LABEL=REPO. The label names the scheme in every table;
# the repo's own quantization_config is what gets recorded, because "AWQ_4B" is
# a name we chose and `bits=4, group_size=128` is what the file asserts.
DEPLOYED_3B = [
    'AWQ_4B=Qwen/Qwen2.5-3B-Instruct-AWQ',
    'GPTQ_4B=Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4',
]
DEPLOYED_7B = [
    'AWQ_4B=Qwen/Qwen2.5-7B-Instruct-AWQ',
    'GPTQ_4B=Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
]
# 22, not 16. A 7B in fp16 is 15.2 GB of weights before any activation, and the
# baseline has to be fp16 or there is nothing to pair the deployed schemes
# against. A T4 cannot hold it, and running the step at a reduced prompt count
# to make it fit would answer a different question -- so it is skipped instead,
# and reported as skipped.
VRAM_FOR_7B_GB = 22.0

# Sampled decoding. Three seeds is not a distribution; it is enough to say
# whether the transition counts move by more than the greedy run-to-run
# variation already measured, which is the question round 3 left open.
SAMPLE_TEMPERATURE, SAMPLE_SEEDS = 0.7, (0, 1, 2)

# What each step must have produced to count as finished. Named here rather
# than discovered from the run directory afterwards: a deployed run that
# silently produced only FP16, because a checkpoint failed to load, would
# otherwise be graded on one scheme and answer H1 with a ratio computed against
# nothing.
SCHEMES_DEPLOYED_3B = ['FP16'] + [spec.split('=')[0] for spec in DEPLOYED_3B]
SCHEMES_DEPLOYED_7B = ['FP16'] + [spec.split('=')[0] for spec in DEPLOYED_7B]
SCHEMES_RTN_4B = ['FP16', 'RTN_4B']

# Verified by preflight; repeated here so a restored run directory whose
# prompts are not the ones we are pairing against is rejected rather than
# reused. The same number of prompts is not the same corpus.
FOLD_A_SHA = '7da25bf88ee0409ce4900a12052e15849a2898ed01cfdcdfe6409bbfc11bd9b5'
XSTEST_SHA = '33874ac77bd574a74283cd024466f442e69da870fa1195fcde8a9107433f9ce4'

CACHE_ROOT = (DRIVE_ROOT / 'artifacts') if DRIVE_ROOT.exists() else pathlib.Path('artifacts')
BEHAV_CACHE = str(CACHE_ROOT / 'behavioural_cache')
RESULTS = {}
STEP_NOTES = {}
print(f'caches: {CACHE_ROOT}' + ('' if DRIVE_ROOT.exists() else '   (LOCAL)'))

def run_step(label, script, args, timeout=10800):
    '''Stream one script invocation; keep its tail and exit status.

    The timeout is a watchdog rather than proc.wait(timeout=...). Reading a
    child's stdout to EOF blocks for as long as it lives, so wait was reached
    only after exit and could never stop a hung step. A watchdog kill is kept
    separate because Linux reports it as -9, the same code as an OOM kill.
    '''
    import threading
    cmd = [sys.executable, f'scripts/{script}'] + args
    print(f'\n$ {" ".join(cmd)}', flush=True)
    started, lines = time.time(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timed_out = []
    def _kill():
        timed_out.append(True)
        print(f'\n[{label}] no exit after {timeout / 3600:.1f} h; killing', flush=True)
        proc.kill()
    watchdog = threading.Timer(timeout, _kill)
    watchdog.daemon = True
    watchdog.start()
    try:
        for line in proc.stdout:
            if 'Loading weights' in line or 'it/s]' in line or 's/prompt' in line:
                continue
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait()
    except Exception as exc:
        proc.kill()
        proc.wait()
        lines.append(f'ABORTED: {type(exc).__name__}: {exc}')
    finally:
        watchdog.cancel()
    ok = proc.returncode == 0
    RESULTS[label] = {'returncode': proc.returncode, 'timed_out': bool(timed_out), 'minutes': (time.time() - started) / 60, 'tail': lines[-40:]}
    print(f'\n=== {label}: {"OK" if ok else f"FAILED rc={proc.returncode}"} in {RESULTS[label]["minutes"]:.1f} min ===', flush=True)
    return ok

def run_step_resumable(label, script, args, attempts=3, timeout=10800):
    '''Retry only a real OOM. Each completed scheme is cached immediately, so a
    fresh process resumes farther through the ladder instead of starting over.
    A bad flag, missing checkpoint, or timeout cannot be improved by retrying.'''
    tag = label
    for attempt in range(1, attempts + 1):
        tag = label if attempt == 1 else f'{label}-retry{attempt}'
        if attempt > 1:
            print(f'\n[retry {attempt}/{attempts}] {label}: resuming from cache', flush=True)
        if run_step(tag, script, args, timeout=timeout):
            RESULTS[label] = RESULTS[tag]
            return True
        result = RESULTS[tag]
        if result['returncode'] != -9 or result['timed_out']:
            reason = 'timed out' if result['timed_out'] else f'rc={result["returncode"]}'
            print(f'[{label}] {reason} is not a resumable OOM; not retrying', flush=True)
            RESULTS[label] = result
            return False
    print(f'[{label}] still failing after {attempts} attempts', flush=True)
    RESULTS[label] = RESULTS[tag]
    return False

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    # GPU allocation is released between schemes; host memory ratchets upward
    # across model loads and is what eventually triggers Colab's OOM killer.
    host = ''
    try:
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                host = f', host available {float(line.split()[1]) / 1e6:.1f} GB'
    except OSError:
        pass
    print(f'[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated{host}')

def checkpoint_to_drive():
    '''Mirror completed run directories to Drive; caches already live there.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ('runs', 'behavioural_cache', 'sector_cache'):
        src = pathlib.Path('artifacts') / name
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / 'artifacts' / name, dirs_exist_ok=True)
    print(f'[drive] mirrored artifacts/ to {DRIVE_ROOT}')

def restore_from_drive():
    '''Bring back prior run directories before anything runs.'''
    if not DRIVE_ROOT.exists():
        return
    import shutil
    src = DRIVE_ROOT / 'artifacts' / 'runs'
    if src.exists():
        shutil.copytree(src, pathlib.Path('artifacts') / 'runs', dirs_exist_ok=True)
        print('[drive] restored artifacts/runs')


def latest_run(pattern):
    hits = sorted(pathlib.Path('artifacts/runs').glob(pattern))
    return hits[-1] if hits else None

def completed_run(label, schemes, model, n_prompts, corpus_sha=None,
                  tokens=None, seed=None, batch_size=None,
                  temperature=None):
    '''A restored run directory that is genuinely finished, not merely present.

    Existence is not completion. A Drive copy interrupted mid-write, a stale
    directory from a run with different arguments, or a truncated JSON all look
    identical to `path.exists()`, and treating any of them as done would skip
    the step and hand the analysis silently wrong data. That is worse than
    re-running: a missing result is visible, a wrong one is not.

    So everything the step depends on is checked against the manifest --
    model, scheme list, prompt count, token budget, seed, and the ordered
    corpus hash that makes the comparison paired at all -- and every
    completions file is parsed and counted rather than stat-ed.
    '''
    tokens = LONG_TOKENS if tokens is None else tokens
    complete = []
    for run in sorted(pathlib.Path('artifacts/runs').glob(f'*_{label}')):
        try:
            manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
        except (OSError, ValueError):
            print(f'[resume] {run.name}: unreadable manifest; ignoring')
            continue
        expected = {'model_id': model, 'n_prompts': n_prompts,
                    'max_new_tokens': tokens,
                    'seed': SEED if seed is None else seed}
        # Checked when the caller says they matter, and for two steps they
        # are the ONLY thing that distinguishes one run from another: the
        # batch-isolation pair differs in nothing else, and a sampled run
        # differs from a greedy one in nothing else. A restored directory
        # from the other arm passes every check above and would be reused,
        # so the step would compare a run against itself.
        if batch_size is not None:
            expected['batch_size'] = batch_size
        if temperature is not None:
            expected['temperature'] = temperature
        wrong = {k: (manifest.get(k), v) for k, v in expected.items()
                 if manifest.get(k) != v}
        if wrong:
            print(f'[resume] {run.name}: arguments differ {wrong}; ignoring')
            continue
        if list(manifest.get('schemes', [])) != schemes:
            print(f'[resume] {run.name}: schemes {manifest.get("schemes")} '
                  f'!= {schemes}; ignoring')
            continue
        digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
        if corpus_sha and digest != corpus_sha:
            print(f'[resume] {run.name}: corpus hash differs; ignoring')
            continue
        ok = True
        for scheme in schemes:
            path = run / 'results' / f'completions_{scheme}.json'
            try:
                blob = json.loads(path.read_text(encoding='utf-8'))
                texts = blob['completions'] if isinstance(blob, dict) else blob
            except (OSError, ValueError, KeyError):
                print(f'[resume] {run.name}: {path.name} missing or unparseable')
                ok = False
                break
            if len(texts) != n_prompts:
                print(f'[resume] {run.name}: {path.name} has {len(texts)} rows, '
                      f'expected {n_prompts}')
                ok = False
                break
        if ok:
            complete.append(run)
    if len(complete) > 1:
        raise SystemExit(f'multiple completed runs share {label}: '
                         f'{[p.name for p in complete]}. Refusing to choose one.')
    return complete[0] if complete else None


def grade_complete(run, filename, schemes, n_prompts):
    '''A grading output that covers every scheme and every prompt.

    Same argument as above: a half-written grade file exists just as much as a
    finished one, and skipping the grader on the strength of that would leave
    the analysis reading rows that were never produced.
    '''
    path = run / 'results' / filename
    try:
        blob = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return False
    # The scorer is part of what makes a grading the right grading. A file
    # written under first-token scoring satisfies every structural check here
    # while being the instrument this round exists to replace, so skipping on
    # its presence would silently keep the old measurement.
    if blob.get('scoring') not in (None, SCORING):
        print(f"[resume] {path.name}: scored under "
              f"{blob.get('scoring')!r}, not {SCORING!r}; will re-grade")
        return False
    results = blob.get('results') or blob.get('verdicts') or {}
    missing = [s for s in schemes if s not in results]
    if missing:
        print(f'[resume] {path.name}: no rows for {missing}; will re-grade')
        return False
    return True


def print_pairing(run):
    """Show the ordered corpus hash, so pairing is visible not assumed."""
    manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
    digest = manifest.get('corpora', {}).get('prompts', {}).get('sha256_ordered')
    print(f'[run] {run}')
    print(f'[pairing] corpora.prompts.sha256_ordered = {digest}')


def record_skip(label, reason):
    RESULTS[label] = {'returncode': 0, 'skipped': True, 'minutes': 0.0,
                      'tail': [reason]}
    print(f'[{label}] already complete after Drive restore; skipping: {reason}')


## Step 1 — deployed quantizers

**H1 and H2 of the frozen protocol.** AWQ and GPTQ-Int4 checkpoints of
Qwen2.5-3B, against the same full-precision baseline, the same 500 HH-RLHF
prompts, the same gate and the same corrected scorer. These carry their
quantization in the weights, so they are loaded rather than constructed, and
the run records what each checkpoint says about itself rather than what its
label implies.

They are not points on the RTN ladder and are never regressed on it: AWQ and
GPTQ change the algorithm, not just the bit-width, so the ordinal axis that
makes a ladder fit meaningful does not contain them.

In [ ]:
# H1 and H2 need the deployed checkpoints AND a protocol to be held to.
# Steps 2-4 need neither, so this skips rather than raising: whether
# SystemExit ends a Colab 'Run all' is a runtime property, and losing three
# working steps to a missing wheel would be a worse failure than the one
# the guard prevents.
# Outside the branch, because it NAMES the step and later cells name it too.
# Inside, a skipped step-1 left it undefined and the verdict cell died with
# NameError -- a correctly skipped step taking the rest of the notebook with it.
STEP1 = 'deployed-3b'

STEP1_BLOCKER = ('preflight failed' if not SESSION_OK else
                 'the frozen protocol is missing or modified' if not PREREG_OK
                 else 'AWQ/GPTQ backends unavailable'
                 if not DEPLOYED_BACKENDS_OK else None)
if STEP1_BLOCKER:
    RESULTS['r5-deployed-3b'] = {'returncode': 1, 'blocked': True,
                                 'minutes': 0.0, 'tail': [STEP1_BLOCKER]}
    STEP_NOTES['deployed_3b'] = {'skipped': True, 'reason': STEP1_BLOCKER}
    run_dir_1 = None
    print(f'SKIPPED: {STEP1_BLOCKER}. H1 and H2 are unanswered; steps 2-4 '
          'below are unaffected and still worth running.')
else:
    # completed_run, not latest_run. A directory left behind by an interrupted
    # session is indistinguishable from a finished one under `is None`, and the
    # whole point of resuming is that a step already done is not repeated -- which
    # is only safe if "done" means every completions file is present and the right
    # length. See the helper's docstring.
    run_dir_1 = completed_run(f'r5-{STEP1}', SCHEMES_DEPLOYED_3B, MODEL_3B, N_LONG * 2,
                              tokens=SHORT_TOKENS, corpus_sha=FOLD_A_SHA)
    if run_dir_1 is None:
        ok = run_step_resumable(
            f'r5-{STEP1}', 'run_behavioural_ladder.py',
            ['--model', MODEL_3B, '--n', str(N_LONG),
             '--bits',                                   # no RTN rungs: FP16 + deployed
             '--deployed', *DEPLOYED_3B,
             '--max-new-tokens', str(SHORT_TOKENS), '--seed', str(SEED),
             '--no-activations', '--cache', BEHAV_CACHE,
             '--label', f'r5-{STEP1}'],
            timeout=60 * 60)
        checkpoint_to_drive()
        run_dir_1 = latest_run(f'*r5-{STEP1}')
    else:
        record_skip(f'r5-{STEP1}', 'complete run directory restored')

    if run_dir_1 is not None:
        print_pairing(run_dir_1)
        schemes = sorted(p.stem.split('_', 1)[1]
                         for p in (run_dir_1 / 'results').glob('completions_*.json'))
        print(f'schemes generated: {schemes}')
        free_vram()
        run_step_resumable(
            f'r5-{STEP1}-judge', 'classify_completions_judge.py',
            [str(run_dir_1), '--judge-model', JUDGE_MODEL, '--judge-4bit',
             '--completion-chars', str(JUDGE_COMPLETION_CHARS),
             '--batch-size', str(JUDGE_BATCH), '--scoring', 'letter',
             '--schemes', *schemes],
            timeout=45 * 60)
        checkpoint_to_drive()
    else:
        print('step 1 produced no run directory; H1 and H2 are unanswered')

## Step 2 — the window, at the quantized rung

The 256-token replication so far covers full precision on XSTest. That
establishes that the empty harmful-compliance cell is not a truncation artifact
*at full precision*; it says nothing about the 21 quantized cells, which are
still 48-token measurements.

This runs the two budgets at the rung the manuscript's surviving claim lives
on — 4.5 stored bits — for both behavioural models and all three labelled ones.
The short side is cut from the long run's own generated token ids, so the two
budgets describe one act of decoding and the window is what differs.

In [ ]:
if not (SESSION_OK):
    print('SKIPPED: preflight failed, so nothing here is run.')
else:
    STEP2 = 'window-4p5'
    window_runs = []
    for tag, model, n_prompts, prompts_arg in (
            ('qwen3b', MODEL_3B, N_LONG, []),
            ('phi35', 'microsoft/Phi-3.5-mini-instruct', N_LONG, []),
            ('qwen3b-xstest', MODEL_3B, N_XSTEST,
             ['--prompts', 'data/eval_suites/xstest.jsonl']),
            ('phi35-xstest', 'microsoft/Phi-3.5-mini-instruct', N_XSTEST,
             ['--prompts', 'data/eval_suites/xstest.jsonl']),
            ('smol17-xstest', 'HuggingFaceTB/SmolLM2-1.7B-Instruct', N_XSTEST,
             ['--prompts', 'data/eval_suites/xstest.jsonl'])):
        label = f'r5-{STEP2}-{tag}'
        long_run = completed_run(label, SCHEMES_RTN_4B, model, n_prompts * 2,
                                 tokens=LONG_TOKENS, corpus_sha=(XSTEST_SHA if 'xstest' in tag else FOLD_A_SHA))
        if long_run is None:
            free_vram()
            run_step_resumable(
                label, 'run_behavioural_ladder.py',
                ['--model', model, '--n', str(n_prompts), '--bits', '4',
                 '--max-new-tokens', str(LONG_TOKENS), '--seed', str(SEED),
                 '--no-activations', '--cache', BEHAV_CACHE, '--label', label,
                 *prompts_arg],
                timeout=60 * 60)
            checkpoint_to_drive()
            long_run = latest_run(f'*{label}')
        else:
            record_skip(label, 'complete run directory restored')
        if long_run is None:
            continue
        # The 48-token side, cut from the long run's OWN token ids rather than
        # generated again. That is what makes the comparison a window comparison
        # instead of two generation passes assumed to agree.
        # make_prefix_run derives its own label from the source run's --
        # "<source>-prefix48" -- so nothing is passed for it here. --compare takes
        # a separately generated short run to check drift against, and there is no
        # such run in this round: the whole point is that the short side is CUT
        # from the long one rather than generated a second time.
        prefix_label = f'{label}-prefix{SHORT_TOKENS}'
        # completed_run here too, and for a sharper reason than elsewhere:
        # make_prefix_run writes its manifest LAST, so a session killed during the
        # NLL scoring pass leaves a directory with completions and no manifest.
        # `latest_run` finds that directory and skips; the grader then opens
        # manifest.json and dies -- two hours in, on a step that had already run.
        prefix_run = completed_run(prefix_label, SCHEMES_RTN_4B, model,
                                   n_prompts * 2, tokens=SHORT_TOKENS, corpus_sha=(XSTEST_SHA if 'xstest' in tag else FOLD_A_SHA))
        if prefix_run is None:
            run_step_resumable(
                f'r5-{STEP2}-{tag}-prefix', 'make_prefix_run.py',
                [str(long_run), '--tokens', str(SHORT_TOKENS)],
                timeout=15 * 60)
            prefix_run = latest_run(f'*{prefix_label}')
            # Immediately, not after the grading below. A prefix run is the one
            # artifact here that is expensive to lose: rebuilding it needs the long
            # run AND a model load for the NLL pass the graders will not run
            # without.
            checkpoint_to_drive()
        if prefix_run is None:
            print(f'[{tag}] no {SHORT_TOKENS}-token prefix run; the window '
                  'comparison for this model cannot be made')
        window_runs.append((tag, long_run, prefix_run))

    # Grade both sides with the same instrument, so the only thing differing
    # between them is the number of tokens the model was allowed.
    for tag, long_run, prefix_run in window_runs:
        grader = ('classify_completion_taxonomy.py' if 'xstest' in tag
                  else 'classify_completions_judge.py')
        for side, run_dir in (('long', long_run), ('short', prefix_run)):
            if run_dir is None:
                continue
            free_vram()
            args = [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
                    '--completion-chars', str(JUDGE_COMPLETION_CHARS),
                    '--batch-size', str(JUDGE_BATCH), '--scoring', 'letter']
            if grader == 'classify_completion_taxonomy.py':
                args += ['--max-length', str(TAXONOMY_MAX_LENGTH)]
            run_step_resumable(f'r5-{STEP2}-{tag}-{side}-judge', grader, args,
                               timeout=45 * 60)
        checkpoint_to_drive()

## Step 3 — is greedy nondeterminism a batch-size effect?

Round 3 found that 9–12% of completions differ between two greedy runs of the
same model on the same prompts, while a third pair reproduced exactly. The two
that diverged were generated at batch 16 and then batch 8; the one that
reproduced was batch 8 both times. Batch shape is the obvious explanation and it
is not the only difference between those runs — corpus, model and library
version all changed too.

This isolates it: one model, one rung, one corpus, one library, two batch sizes.
No grading, because the question is whether the *text* differs. Twelve minutes,
and it converts a plausible mechanism into a measured one.

In [ ]:
if not (SESSION_OK):
    print('SKIPPED: preflight failed, so nothing here is run.')
else:
    STEP3 = 'batch-isolation'
    batch_runs = {}
    for batch in (8, 16):
        label = f'r5-{STEP3}-b{batch}'
        done = completed_run(label, SCHEMES_RTN_4B, MODEL_3B, N_LONG * 2,
                             tokens=SHORT_TOKENS, corpus_sha=FOLD_A_SHA, batch_size=batch)
        if done is None:
            free_vram()
            run_step_resumable(
                label, 'run_behavioural_ladder.py',
                ['--model', MODEL_3B, '--n', str(N_LONG), '--bits', '4',
                 '--max-new-tokens', str(SHORT_TOKENS), '--seed', str(SEED),
                 '--batch-size', str(batch), '--no-activations',
                 '--cache', BEHAV_CACHE, '--label', label],
                timeout=45 * 60)
            checkpoint_to_drive()
            done = latest_run(f'*{label}')
        else:
            record_skip(label, 'complete run directory restored')
        batch_runs[batch] = done

    # Compared here rather than in a script, because the comparison is three lines
    # and the answer belongs on the screen while the runtime is still attached.
    if all(batch_runs.values()):
        divergence = {}
        for scheme in ('FP16', 'RTN_4B'):
            texts = []
            for batch in (8, 16):
                path = batch_runs[batch] / 'results' / f'completions_{scheme}.json'
                if not path.exists():
                    break
                texts.append(json.loads(path.read_text(encoding='utf-8'))['completions'])
            if len(texts) != 2 or len(texts[0]) != len(texts[1]):
                continue
            differ = sum(a != b for a, b in zip(*texts))
            divergence[scheme] = {'differ': differ, 'n': len(texts[0]),
                                  'share': differ / len(texts[0])}
            print(f'{scheme:8s} batch 8 vs 16: {differ}/{len(texts[0])} '
                  f'completions differ ({100 * differ / len(texts[0]):.1f}%)')
        STEP_NOTES['batch_isolation'] = divergence
        if divergence:
            worst = max(v['share'] for v in divergence.values())
            print('\n-> batch size alone ' +
                  ('reproduces the 9-12% divergence, so it is the mechanism'
                   if worst >= 0.05 else
                   'does NOT reproduce it; something else in those runs differed'))
    else:
        print('step 3 incomplete; the mechanism stays a conjecture')

## Step 4 — sampled decoding across seeds

Every result in this project is greedy. That is a defensible choice — it removes
one source of variance and makes a paired comparison exact — and it means
nothing here says how the transition counts behave when a deployment samples.

Three seeds at temperature 0.7, at full precision and at 4.5 bits, on
Qwen2.5-3B. Three seeds is not a distribution. It is enough to say whether the
spread across samples is larger than the greedy run-to-run variation already
measured, which is the question that matters for reading any of the paper's
point estimates.

In [ ]:
if not (SESSION_OK):
    print('SKIPPED: preflight failed, so nothing here is run.')
else:
    STEP4 = 'sampled'
    sampled_runs = []
    for seed in SAMPLE_SEEDS:
        label = f'r5-{STEP4}-s{seed}'
        # Note the seed: completed_run checks it against the manifest, so a restored
        # directory from a different seed is rejected rather than reused. Under
        # sampling that is not pedantry -- the whole step is about what changes
        # between seeds.
        done = completed_run(label, SCHEMES_RTN_4B, MODEL_3B, N_LONG * 2,
                             tokens=SHORT_TOKENS, seed=seed, corpus_sha=FOLD_A_SHA, temperature=SAMPLE_TEMPERATURE)
        if done is None:
            free_vram()
            run_step_resumable(
                label, 'run_behavioural_ladder.py',
                ['--model', MODEL_3B, '--n', str(N_LONG), '--bits', '4',
                 '--max-new-tokens', str(SHORT_TOKENS),
                 '--temperature', str(SAMPLE_TEMPERATURE), '--seed', str(seed),
                 '--no-activations', '--cache', BEHAV_CACHE, '--label', label],
                timeout=45 * 60)
            checkpoint_to_drive()
            done = latest_run(f'*{label}')
        else:
            record_skip(label, 'complete run directory restored')
        run_dir = done
        if run_dir is None:
            continue
        free_vram()
        run_step_resumable(
            f'{label}-judge', 'classify_completions_judge.py',
            [str(run_dir), '--judge-model', JUDGE_MODEL, '--judge-4bit',
             '--completion-chars', str(JUDGE_COMPLETION_CHARS),
             '--batch-size', str(JUDGE_BATCH), '--scoring', 'letter'],
            timeout=45 * 60)
        sampled_runs.append(run_dir)
        checkpoint_to_drive()
    print(f'\nsampled runs completed: {len(sampled_runs)}/{len(SAMPLE_SEEDS)}')

## Step 5 — scale, if the runtime carries it

Four models, none above 3.8B. One 7B does not make a scaling study; it checks
that nothing about the protocol breaks at a size where deployment actually
quantizes, and it is the smallest step that answers the objection at all.

A 7B in fp16 is 15.2 GB of weights before any activation, and the baseline has
to be fp16 or there is nothing to pair the deployed schemes against. On a T4
this step skips itself and says so. It is **not** run at a reduced prompt count
to make it fit, because that would answer a different question and report it
under the same name.

In [ ]:
STEP5 = 'deployed-7b'
if not (SESSION_OK and PREREG_OK):
    STEP_NOTES['scale'] = {'skipped': True, 'reason': 'session not sound'}
    print('SKIPPED: preflight failed or the frozen protocol is not intact.')
elif not DEPLOYED_BACKENDS_OK:
    STEP_NOTES['scale'] = {'skipped': True,
                           'reason': 'AWQ/GPTQ backends unavailable'}
    print('SKIPPED: the AWQ/GPTQ backends are not available, so the 7B arm '
          'has nothing to load. H4 is unanswered.')
elif VRAM_GB < VRAM_FOR_7B_GB:
    STEP_NOTES['scale'] = {'skipped': True, 'vram_gb': VRAM_GB,
                           'needed_gb': VRAM_FOR_7B_GB}
    print(f'SKIPPED: {VRAM_GB} GB of VRAM, {VRAM_FOR_7B_GB} needed for a 7B '
          f'fp16 baseline.\nH4 is unanswered. Runtime -> Change runtime type '
          '-> L4 or A100 and rerun this cell to answer it.')
else:
    run_dir_5 = completed_run(f'r5-{STEP5}', SCHEMES_DEPLOYED_7B, MODEL_7B,
                              N_LONG * 2, tokens=SHORT_TOKENS, corpus_sha=FOLD_A_SHA)
    if run_dir_5 is None:
        free_vram()
        run_step_resumable(
            f'r5-{STEP5}', 'run_behavioural_ladder.py',
            ['--model', MODEL_7B, '--n', str(N_LONG), '--bits',
             '--deployed', *DEPLOYED_7B,
             '--max-new-tokens', str(SHORT_TOKENS), '--seed', str(SEED),
             '--no-activations', '--cache', BEHAV_CACHE,
             '--label', f'r5-{STEP5}'],
            timeout=90 * 60)
        checkpoint_to_drive()
        run_dir_5 = latest_run(f'*r5-{STEP5}')
    else:
        record_skip(f'r5-{STEP5}', 'complete run directory restored')

    if run_dir_5 is not None:
        schemes = sorted(p.stem.split('_', 1)[1]
                         for p in (run_dir_5 / 'results').glob('completions_*.json'))
        free_vram()
        run_step_resumable(
            f'r5-{STEP5}-judge', 'classify_completions_judge.py',
            [str(run_dir_5), '--judge-model', JUDGE_MODEL, '--judge-4bit',
             '--completion-chars', str(JUDGE_COMPLETION_CHARS),
             '--batch-size', str(JUDGE_BATCH), '--scoring', 'letter',
             '--schemes', *schemes],
            timeout=60 * 60)
        checkpoint_to_drive()

## What the frozen protocol predicted, and what happened

H1 and H2 have thresholds fixed before the run. This reads them off the stored
results and prints the verdict, so the comparison between prediction and outcome
is made by arithmetic rather than by whoever writes the manuscript afterwards.

In [ ]:
verdicts = {}
if not PREREG_OK:
    print('No intact preregistration, so there is nothing to score '
          'these results against. H1 and H2 are unanswerable in this '
          'session whatever the numbers say.')
run_dir_1 = latest_run(f'*r5-{STEP1}')
verdict_path = pathlib.Path('artifacts/runs/r5_deployed.json')
# Deleted before the attempt, not after it. The alternative -- run, then read
# the file if it exists -- serves the PREVIOUS session's verdict whenever this
# one fails, and this is the single number the protocol fixed in advance. A
# stale CONFIRMED is worse than no verdict at all.
verdict_path.unlink(missing_ok=True)

if run_dir_1 is None:
    print('step 1 did not run; H1 and H2 are unanswered')
else:
    out = subprocess.run(
        [sys.executable, 'scripts/analyse_deployed.py', str(run_dir_1),
         '--scorer', 'letter', '--judge', JUDGE_MODEL,
         '--out', str(verdict_path)],
        capture_output=True, text=True)
    print(out.stdout or out.stderr)
    if out.returncode != 0:
        RESULTS['r5-prereg-verdict'] = {
            'returncode': out.returncode, 'minutes': 0.0,
            'tail': (out.stderr or out.stdout).splitlines()[-20:]}
        print(f'\nanalyse_deployed exited {out.returncode}: H1 and H2 have no '
              'verdict from this session.')
    elif verdict_path.exists():
        verdicts = json.loads(verdict_path.read_text(encoding='utf-8'))
        print(f"\nH1 {verdicts.get('h1')}   H2 {verdicts.get('h2')}")
STEP_NOTES['prereg_verdicts'] = verdicts

## Export

The archive carries every run directory this round produced, the preregistration
and its hash, and the step ledger. As in earlier rounds the filename is prefixed
`INCOMPLETE_` if any step failed or never ran, because a step that produced
nothing is not a step that found nothing.

In [ ]:
import zipfile

stamp = time.strftime('%Y%m%d-%H%M%S')
failed = [k for k, v in RESULTS.items()
          if v.get('returncode') not in (0, None) or v.get('blocked')]
prefix = 'INCOMPLETE_' if failed else ''
name = f'{prefix}cliffguard_round5_{stamp}.zip'
archive = pathlib.Path(f'/content/{name}' if IN_COLAB else name)

status = {'steps': RESULTS, 'failed': failed, 'notes': STEP_NOTES,
          'preregistration_sha256': PREREG_SHA,
          'scoring': 'letter', 'batch_size': JUDGE_BATCH,
          'deployed_3b': DEPLOYED_3B, 'deployed_7b': DEPLOYED_7B,
          'short_tokens': SHORT_TOKENS, 'long_tokens': LONG_TOKENS,
          'sample_temperature': SAMPLE_TEMPERATURE,
          'sample_seeds': list(SAMPLE_SEEDS),
          'vram_gb': VRAM_GB, 'gpu': GPU_NAME}

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for run_dir in sorted(pathlib.Path('artifacts/runs').glob('*r5-*')):
        for path in sorted(run_dir.rglob('*')):
            if not path.is_file() or 'activations' in path.parts:
                continue
            zf.write(path, path.as_posix())
    for extra in sorted(pathlib.Path('artifacts/runs').glob('r5_*.json')):
        zf.write(extra, extra.as_posix())
    if PREREG.exists():
        zf.write(PREREG, PREREG.as_posix())
    zf.writestr('artifacts/runs/ROUND5_STATUS.json', json.dumps(status, indent=2))

print(f'wrote {archive}  ({archive.stat().st_size / 1e6:.1f} MB)')
print('\nSTEPS:')
for key, value in RESULTS.items():
    mark = 'ok' if value.get('returncode') == 0 else f'FAIL({value.get("returncode")})'
    suffix = ' (skipped)' if value.get('skipped') else ''
    print(f'  {mark:9s} {key:34s} {value.get("minutes", 0):6.1f} min{suffix}')
if failed:
    print(f'\nINCOMPLETE: {failed}')
    print('Do not report a step that did not run as a step that found nothing.')

try:
    shutil.copy2(archive, DRIVE_ROOT / archive.name)
    print(f'\ncopied to Drive: {archive.name}')
except Exception as error:
    print(f'\ncould not copy to Drive ({error}); download it from /content')

try:
    from google.colab import files
    files.download(str(archive))
except Exception:
    pass